# Data Exploration

Run after `scripts/01_prepare_split.py`. Requires `data/manifest.csv`.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import SimpleITK as sitk
from pathlib import Path

MANIFEST = '../data/manifest.csv'
df = pd.read_csv(MANIFEST)
print(f'Total rows: {len(df)}')
print(df.head())

In [ ]:
# Split distribution
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Patients per split
pat_counts = df.drop_duplicates('patient_id').groupby('split').size()
axes[0].bar(pat_counts.index, pat_counts.values, color=['steelblue','orange','green'])
axes[0].set_title('Patients per split')
axes[0].set_ylabel('Count')

# Quality distribution
qual_split = df.drop_duplicates('patient_id').groupby(['split','quality']).size().unstack(fill_value=0)
qual_split.T.plot(kind='bar', ax=axes[1])
axes[1].set_title('Quality by split')
axes[1].tick_params(axis='x', rotation=0)

# Images per view x phase
vp = df.groupby(['view','phase']).size().unstack()
vp.plot(kind='bar', ax=axes[2])
axes[2].set_title('Images by view/phase')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Sample overlays — 6 random training cases
sample = df[df['split'] == 'train'].sample(6, random_state=42)

cmap = plt.get_cmap('tab10')
fig, axes = plt.subplots(2, 6, figsize=(18, 6))

for col, (_, row) in enumerate(sample.iterrows()):
    img_arr = sitk.GetArrayFromImage(sitk.ReadImage(row['image']))
    gt_arr  = sitk.GetArrayFromImage(sitk.ReadImage(row['mask']))
    if img_arr.ndim == 3:
        img_arr, gt_arr = img_arr[0], gt_arr[0]
    img_arr = (img_arr - img_arr.min()) / (img_arr.max() - img_arr.min() + 1e-8)

    axes[0, col].imshow(img_arr, cmap='gray')
    axes[0, col].set_title(f"{row['patient_id']}\n{row['view']} {row['phase']} ({row['quality']})", fontsize=7)
    axes[0, col].axis('off')

    axes[1, col].imshow(img_arr, cmap='gray')
    axes[1, col].imshow(gt_arr, alpha=0.4, cmap=cmap, vmin=0, vmax=3)
    axes[1, col].axis('off')

patches = [mpatches.Patch(color=cmap(i), label=l) for i, l in enumerate(['bg','LV','MYO','LA'])]
fig.legend(handles=patches, loc='lower center', ncol=4, fontsize=9)
plt.suptitle('Top: image  |  Bottom: image + GT mask', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Image statistics: pixel spacing and image size
spacings, heights, widths = [], [], []
for _, row in df[df['split'] == 'train'].iterrows():
    img = sitk.ReadImage(row['image'])
    sp = img.GetSpacing()
    sz = img.GetSize()
    spacings.append((sp[0], sp[1]))
    widths.append(sz[0])
    heights.append(sz[1])

spacings = np.array(spacings)
print(f'Pixel spacing (mm) — x: {spacings[:,0].min():.3f}–{spacings[:,0].max():.3f}  '
      f'y: {spacings[:,1].min():.3f}–{spacings[:,1].max():.3f}')
print(f'Image size — W: {min(widths)}–{max(widths)}  H: {min(heights)}–{max(heights)}')